# 00 — Overview

Welcome to the analysis suite.  Every notebook here reads the CSV
reports written by `Experiment.run()` under `reports/<exp_id>/`:

* `episode_metrics.csv` — one row per episode (training, eval, …).
* `step_metrics.csv`    — one row per env step.
* `config.json`         — the env + agent + experiment config snapshot.

This first notebook shows:

1. Which runs are available on disk.
2. How to load one.
3. The high-level structure of the resulting DataFrames.
4. A one-glance summary of the run.

In [1]:
import sys, pathlib
# Make ``utils.py`` importable whether the notebook is opened from
# project root or from analysis/.
_here = pathlib.Path.cwd()
for cand in (_here, _here / "analysis", _here.parent):
    if (cand / "utils.py").exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    list_runs, latest_run, load_run, load_runs,
    group_columns, group_columns_by_prefix, strip_prefix, rolling_mean,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## Available runs

In [2]:
runs_df = list_runs()
runs_df

,exp_id,has_episodes,has_steps,has_config,modified,path
0,nb_least_busy,True,True,True,2026-05-11 09:34:58.594967604,/Users/adrien.bolling/repositories/knowledge-a...
1,nb_random,True,True,True,2026-05-11 09:34:58.381193399,/Users/adrien.bolling/repositories/knowledge-a...


## Load the latest run

In [3]:
# Pick a run.  ``latest_run()`` returns the most recently modified
# experiment in ``reports/``.  Override ``EXP_ID = "..."`` to analyse
# a specific one.
EXP_ID = latest_run()
print("Analysing:", EXP_ID)

ep, st, cfg = load_run(EXP_ID)
print(f"  episode_metrics.csv: shape={ep.shape}")
print(f"  step_metrics.csv:    shape={st.shape if st is not None else 'absent'}")

Analysing: nb_least_busy
  episode_metrics.csv: shape=(3, 104)
  step_metrics.csv:    shape=(75, 49)


## Episode metrics — columns by prefix

The dataframe is wide.  Columns are namespaced with a prefix that
tells you what they are:

| Prefix | Meaning |
|---|---|
| `metric/` | Episode-level KPIs (`metric/mttr`, `metric/total_breakdowns`, …) |
| `step_mean/` | Mean of each per-step metric over the episode |
| `reward_sum/` / `reward_mean/` | Total + mean per-step reward components |
| `update/` | Agent-update metrics (PPO loss, KL, …) |
| `assignments/` | Per-tech repair-assignment count |
| `machine_maintenance/` / `machine_production/` / `machine_breakdowns/` | Per-machine episode stats |

In [4]:
groups = group_columns_by_prefix(ep)
for prefix in sorted(groups):
    print(f"  {prefix + '/':<25} {len(groups[prefix])} columns")

  assignments/              4 columns
  machine_breakdowns/       15 columns
  machine_maintenance/      15 columns
  machine_production/       15 columns
  metric/                   8 columns
  reward_mean/              11 columns
  reward_sum/               11 columns
  step_mean/                15 columns


## Step metrics — head

In [5]:
if st is not None and not st.empty:
    display(st.head(5))
else:
    print("no step_metrics.csv for this run")

,phase,episode,step,seed,sim_time,pending_queue_size,has_open_ticket,ticket_machine_id,ticket_machine_type,ticket_machine_name,ticket_component_type,ticket_created_at,ticket_wait_time,action,reward,repair_time_delta,repair_time_delta_per,repair_quality,tech_knowledge/expert_1,tech_knowledge/senior_1,tech_knowledge/generalist_1,tech_knowledge/junior_1,tech_fatigue/expert_1,tech_fatigue/senior_1,tech_fatigue/generalist_1,tech_fatigue/junior_1,tech_specialization/expert_1,tech_specialization/senior_1,tech_specialization/generalist_1,tech_specialization/junior_1,reward_assignment,reward_wait_time,reward_queue_size,reward_busy_technician,reward_fatigue_cost,reward_fleet_knowledge,reward_estimated_repair_time,reward_fleet_availability,reward_throughput_delta,reward_downtime_cost,reward_selection_diversity,total_breakdowns,total_assignments,total_repairs,ill_technician_count,finished_products,mttr,fleet_availability_rate,throughput_rate
0,train,1,1,1,129.0,0,True,3195,Conveyor,conv_1,mechanical,93.0,0.0,0,1.933333,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,-0.0,-0.0,0.0,-0.0,0.000000,-0.0,0.933333,0.0,-0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,train,1,2,1,268.0,0,True,4856,Assembly,asm_3,end_effector,129.0,0.0,0,0.838760,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,-0.0,-0.0,0.0,-0.0,0.000000,-0.0,0.866667,0.0,-0.027907,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,train,1,3,1,503.0,0,True,9415,Conveyor,conv_2,mechanical,268.0,0.0,0,0.682836,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,-0.0,-0.0,0.0,-0.0,0.000000,-0.0,0.800000,0.0,-0.117164,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,train,1,4,1,589.0,0,True,359,Assembly,asm_1,motor,503.0,0.0,1,1.722823,0.0,0.0,0.0,1.567216,0.0,0.0,0.0,0.277473,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,-0.0,-0.0,0.0,-0.0,0.058741,-0.0,0.866667,0.0,-0.202584,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,train,1,5,1,628.0,0,True,9415,Conveyor,conv_2,mechanical,589.0,0.0,3,1.841910,0.0,0.0,0.0,4.754277,0.0,0.0,0.0,0.378115,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,-0.0,-0.0,0.0,-0.0,0.177451,-0.0,0.866667,0.0,-0.202207,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## High-level summary

In [6]:
summary = (
    ep.groupby("phase")
      .agg(
          n_episodes=("episode", "count"),
          mean_return=("return", "mean"),
          last_return=("return", "last"),
          mean_length=("length", "mean"),
          total_wall_clock_s=("wall_clock_seconds", "sum"),
      )
      .round(3)
)
summary

,n_episodes,mean_return,last_return,mean_length,total_wall_clock_s
phase,,,,,
train,3,46.909,51.763,25.0,0.192


## Config snapshot

In [7]:
print("agent:    ", cfg.get("agent", {}).get("agent_type"))
print("seed:     ", cfg.get("experiment", {}).get("seed"))
print("n_eps:    ", cfg.get("experiment", {}).get("n_episodes"))
print("obs_repr: ", cfg.get("env", {}).get("gym", {}).get("observation_representation"))
print("obs_mode: ", cfg.get("env", {}).get("gym", {}).get("observation_mode"))
print("randomized:", cfg.get("env", {}).get("randomized_scenario", {}).get("enabled"))

agent:     least_busy
seed:      0
n_eps:     3
obs_repr:  structured
obs_mode:  factory_level
randomized: False
